In [2]:
import torch 


In [3]:
D_OBS = 1  # input channels (gray scale) 
H_ENC = 32 # hidden dim in encoder 
D_STC = 16 # represenation dim (encoder output channels) 

# Step 1 : ResNet 5 

involves 
- ResNet5 
- TemporalBatchMixin
- ResidualBlock

In [4]:
from architectures import ResNet5

In [5]:
from einops import rearrange 

observations = torch.randn(32, 1, 10, 64, 64)   # Video 
observations = rearrange(observations, "b c t h w -> (b t) c h w")
print(observations.shape)  # notice we have treated each video (with 10 frames) as 10 images ( 32 x 10 ) = 320 images. 

torch.Size([320, 1, 64, 64])


In [6]:
encoder = ResNet5(D_OBS, H_ENC, D_STC)

In [7]:
with torch.no_grad():
    state = encoder._forward(observations)   # [(B,T), C=D_STC=16, H, W]
    print(state.shape)

# after we finished going through encoder we will rearrange images back to videos 
state = rearrange(state, "(b t) c h w -> b c t h w", b = 32)
print(f'Video: {state.shape}')

torch.Size([320, 16, 64, 64])
Video: torch.Size([32, 16, 10, 64, 64])


### Output shape 

In [8]:
print(state.shape)

torch.Size([32, 16, 10, 64, 64])


# unroll_mode `parallel`

for _ in range(nsteps): 

### Step 2 : StateOnlyPredictor

involves 

- StateOnlyPredictor
- SimplePredictor
- ResUNet

In [9]:
predicted_states = state
print(predicted_states.shape)

torch.Size([32, 16, 10, 64, 64])


In [10]:
from architectures import StateOnlyPredictor, ResUNet
H_PRE = 32   # hidden dim in predictor
action_encoded = None 

In [11]:
predictor = StateOnlyPredictor(
    predictor=  ResUNet(2 * D_STC, H_PRE, D_STC), 
    context_length= 2 
)

print(getattr(predictor, "context_length")) 
print(predictor.context_length)

2
2


Output shape 

In [12]:
with torch.no_grad():
    predicted_states = predictor(predicted_states,a=action_encoded)[:,:,:-1]
    print(predicted_states.shape)

torch.Size([32, 16, 8, 64, 64])


### Step 3: Refeed Ground Truth context on the left 

In [13]:


predicted_states = torch.cat(
    (state[:, :, :predictor.context_length],predicted_states),dim=2
)

print(predicted_states.shape)

torch.Size([32, 16, 10, 64, 64])


# Step 3 : Regularizer 

involves 

- VCLoss

# Step 5: Loss 

In [14]:
print(state.shape )
print(predicted_states.shape)

torch.Size([32, 16, 10, 64, 64])
torch.Size([32, 16, 10, 64, 64])


In [15]:
flatten_state = state.transpose(0,1).flatten(1).transpose(0,1)
predicted_states_flatten = predicted_states.transpose(0,1).flatten(1).transpose(0,1)

In [16]:
print(flatten_state.shape)
print(predicted_states_flatten.shape)

torch.Size([1310720, 16])
torch.Size([1310720, 16])


There might be a projector layer before calculating the loss 

In [17]:
from architectures import Projector 

projector = Projector(f"{D_STC}-{D_STC*4}-{D_STC*4}")
projector

Projector(
  (net): Sequential(
    (0): Linear(in_features=16, out_features=64, bias=True)
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Linear(in_features=64, out_features=64, bias=False)
  )
)

In [19]:
with torch.no_grad(): 
    print(projector(flatten_state).shape)
    print(projector(predicted_states_flatten).shape)


torch.Size([1310720, 64])
torch.Size([1310720, 64])
